# 🐍 Python `def`: O Guia Completo e Definitivo de Funções

Um mergulho aprofundado na palavra-chave `def`, cobrindo desde o comportamento interno em runtime e anatomia de parâmetros até regras de escopo (LEGB), closures, introspecção e padrões de design.

---

## 📑 Sumário
1. [O que é `def`? (Execução em Runtime & Bytecode)](#1.-O-que-%C3%A9-def%3F-(Execu%C3%A7%C3%A3o-em-Runtime-&-Bytecode))
2. [Anatomia dos Parâmetros e Argumentos](#2.-Anatomia-dos-Par%C3%A2metros-e-Argumentos)
   - 2.1 Posicionais vs Nomeados
   - 2.2 Valores Padrão (Defaults)
   - 2.3 🚨 A Armadilha Clássica do Default Mutável (`def f(lista=[])`)
   - 2.4 Positional-Only (`/`) e Keyword-Only (`*`)
   - 2.5 Empacotamento e Desempacotamento (`*args` e `**kwargs`)
   - 2.6 A Ordem Canônica de Assinatura
3. [Type Hints e Documentação (PEP 484 & Docstrings)](#3.-Type-Hints-e-Documenta%C3%A7%C3%A3o-(PEP-484-&-Docstrings))
4. [Resolução de Escopo: A Regra LEGB](#4.-Resolu%C3%A7%C3%A3o-de-Escopo:-A-Regra-LEGB)
   - 4.1 Local, Enclosing, Global, Built-in
   - 4.2 Modificadores `global` e `nonlocal`
5. [Funções como Cidadãos de Primeira Classe (*First-Class Objects*)](#5.-Fun%C3%A7%C3%B5es-como-Cidad%C3%A3os-de-Primeira-Classe)
   - 5.1 Funções em Dicionários (Pattern Dispatcher)
   - 5.2 Funções de Alta Ordem (*Higher-Order Functions*)
6. [Funções Aninhadas e Closures](#6.-Fun%C3%A7%C3%B5es-Aninhadas-e-Closures)
   - 6.1 Anatomia de uma Closure
   - 6.2 Fábricas de Funções (*Function Factories*)
7. [Introspecção e Metadados de Funções](#7.-Introspec%C3%A7%C3%A3o-e-Metadados-de-Fun%C3%A7%C3%B5es)
   - 7.1 Atributos Dunder (`__name__`, `__defaults__`, `__code__`)
   - 7.2 O Módulo `inspect`
8. [Recursão e Limite de Pilha](#8.-Recurs%C3%A3o-e-Limite-de-Pilha)
9. [Tabela Resumo e Melhores Práticas](#9.-Tabela-Resumo-e-Melhores-Pr%C3%A1ticas)

## 1. O que é `def`? (Execução em Runtime & Bytecode)

### 💡 Conceito
Ao contrário de linguagens como C ou Java onde funções são declaradas estaticamente, em Python **`def` é uma instrução executável**.

Quando o interpretador encontra `def nome_funcao(...):`:
1. O corpo é compilado em um objeto de código imutável (`code object`).
2. É criado um objeto da classe `types.FunctionType` na memória heap.
3. O identificador `nome_funcao` é vinculado (*bound*) a essa instância no namespace atual.

In [ ]:
import dis

# 1.1 Definição condicional em runtime
MODO_DEBUG = True

if MODO_DEBUG:
    def registrar(evento):
        print(f'[DEBUG-ON] {evento}')
else:
    def registrar(evento):
        pass  # Sem overhead em produção

registrar('Usuário autenticado com sucesso')

# 1.2 Inspecionando o bytecode gerado pelo compilador Python
def soma_simples(a, b):
    return a + b

print('\n--- Desmontagem do Bytecode (dis.dis) ---')
dis.dis(soma_simples)

## 2. Anatomia dos Parâmetros e Argumentos

Python oferece flexibilidade total para passagem de argumentos: posicionais, nomeados, valores padrão, restrições e coleções arbitrárias.

In [ ]:
# 2.1 Posicionais vs Nomeados (Keywords)
def formatar_perfil(nome, cargo, empresa='Freelancer'):
    return f'{nome} - {cargo} na {empresa}'

# Passagem posicional (ordem determina o vínculo)
print(formatar_perfil('Lucas', 'Engenheiro de Software', 'Google'))

# Passagem nomeada (ordem irrelevante)
print(formatar_perfil(empresa='OpenAI', cargo='Pesquisador', nome='Lucas'))

In [ ]:
# 2.2 e 2.3 🚨 A ARMADILHA CLÁSSICA DO DEFAULT MUTÁVEL
# Parâmetros padrão são avaliados UMA ÚNICA VEZ quando a função é DEFINIDA!

# ❌ PERIGO: Lista compartilhada entre chamadas
def adicionar_carrinho_errado(item, carrinho=[]):
    carrinho.append(item)
    return carrinho

print('Chamada 1 (Errado):', adicionar_carrinho_errado('Teclado'))
print('Chamada 2 (Errado):', adicionar_carrinho_errado('Mouse'))  # Opa! Teclado ainda está lá!

# ✅ PADRÃO IDIOMÁTICO: Default imutável (None) + instanciação interna
def adicionar_carrinho_correto(item, carrinho=None):
    if carrinho is None:
        carrinho = []
    carrinho.append(item)
    return carrinho

print('\nChamada 1 (Correto):', adicionar_carrinho_correto('Teclado'))
print('Chamada 2 (Correto):', adicionar_carrinho_correto('Mouse'))  # Nova lista independente!

In [ ]:
# 2.4 Positional-Only (/) e Keyword-Only (*)
#   / -> Tudo ANTES da barra DEVE ser posicional
#   * -> Tudo DEPOIS do asterisco DEVE ser nomeado

def configurar_servidor(ip, porta, /, protocolo='HTTP', *, timeout=30, ssl=True):
    return {
        'endereco': f'{protocolo.lower()}://{ip}:{porta}',
        'timeout': timeout,
        'ssl': ssl
    }

# Chamada correta:
server = configurar_servidor('192.168.1.10', 8080, 'HTTPS', timeout=15, ssl=True)
print('Servidor configurado:', server)

# 2.5 Empacotamento e Desempacotamento (*args e **kwargs)
def pipeline_dados(etapa_inicial, *transformacoes, debug=False, **metadados):
    print(f'\n[Pipeline] Início: {etapa_inicial}')
    print(f'  Transformações (*args - {type(transformacoes).__name__}): {transformacoes}')
    print(f'  Metadados (**kwargs - {type(metadados).__name__}): {metadados}')

pipeline_dados(
    'Ingestao',
    'Normalizacao', 'Filtragem', 'Agrupamento',
    debug=True,
    autor='Lucas',
    versao='2.4.0'
)

### 2.6 A Ordem Canônica de Parâmetros em Python

Ao construir assinaturas completas, a ordem obrigatória imposta pela gramática do Python é:

$$\text{def func}(\underbrace{\text{pos\_only}}_{/},\ \underbrace{\text{padrao\_pos\_ou\_kw}},\ \underbrace{\text{*args}},\ \underbrace{\text{kw\_only}}_{*},\ \underbrace{\text{**kwargs}}):$$

```python
def assinatura_completa(a, b, /, c, d=10, *args, e, f=20, **kwargs):
    pass
```

## 3. Type Hints e Documentação (PEP 484 & Docstrings)

O ecossistema moderno de Python utiliza *type hints* para verificação estática (Mypy, Pyright) e docstrings padronizadas (Google Style / NumPy Style).

In [ ]:
from typing import Optional, Union, List, Callable

def processar_transacao(
    valor: float,
    moeda: str = 'BRL',
    taxa_conversao: Optional[float] = None,
    callbacks: Optional[List[Callable[[float], None]]] = None
) -> float:
    """Calcula o valor líquido de uma transação financeira.

    Args:
        valor (float): Montante bruto da transação.
        moeda (str, optional): Código ISO da moeda. Padrão: 'BRL'.
        taxa_conversao (Optional[float]): Taxa cambial aplicável. Padrão: None (1.0).
        callbacks (Optional[List[Callable]]): Lista de funções receptoras.

    Returns:
        float: Valor final convertido e processado.

    Raises:
        ValueError: Se o valor for negativo.
    """
    if valor < 0:
        raise ValueError('O valor não pode ser negativo.')
    
    taxa = taxa_conversao if taxa_conversao is not None else 1.0
    liquido = valor * taxa
    
    if callbacks:
        for cb in callbacks:
            cb(liquido)
            
    return liquido

print('Anotações de tipo salvas no objeto:', processar_transacao.__annotations__)
print('Docstring formatada:\n', processar_transacao.__doc__.strip()[:180] + '...')

## 4. Resolução de Escopo: A Regra LEGB

Ao referenciar qualquer variável dentro de um corpo de função `def`, o Python busca os identificadores sequencialmente através de 4 níveis de escopo:

```
 ┌─────────────────────────────────────────────────────────────┐
 │ L -> Local     : Variáveis declaradas dentro da função      │
 │ E -> Enclosing : Variáveis de funções aninhadas pai (extern)│
 │ G -> Global    : Variáveis no nível do módulo (arquivo)     │
 │ B -> Built-in  : Nomes nativos da linguagem (len, int, abs) │
 └─────────────────────────────────────────────────────────────┘
```

In [ ]:
# 4.1 Visualizando a hierarquia LEGB
nivel = 'GLOBAL (Módulo)'

def escopo_externo():
    nivel = 'ENCLOSING (Função Pai)'
    
    def escopo_interno():
        nivel = 'LOCAL (Função Filha)'
        print(f'1. Interno acessa: {nivel}')
        
    escopo_interno()
    print(f'2. Externo acessa: {nivel}')

escopo_externo()
print(f'3. Raiz acessa: {nivel}')

In [ ]:
# 4.2 Alterando escopos com 'global' e 'nonlocal'
contador_global = 0

def incrementar_global():
    global contador_global
    contador_global += 10

incrementar_global()
print('Contador global após chamada:', contador_global)

# 'nonlocal' em funções aninhadas (Mantém estado em closures)
def criar_acumulador(saldo_inicial=0):
    saldo = saldo_inicial  # Enclosing
    
    def depositar(quantia):
        nonlocal saldo  # Modifica a variável da função pai, sem sujar o escopo global
        saldo += quantia
        return saldo
        
    return depositar

conta = criar_acumulador(100)
print('\nSaldo pós depósito 50:', conta(50))
print('Saldo pós depósito 30:', conta(30))

## 5. Funções como Cidadãos de Primeira Classe (*First-Class Objects*)

Em Python, funções criadas com `def` são instâncias de primeira classe. Elas podem ser:
- Atribuídas a variáveis;
- Armazenadas em estruturas de dados (listas, dicionários);
- Passadas como argumentos para outras funções;
- Retornadas como valor de outras funções.

In [ ]:
# 5.1 Pattern Dispatcher: Substituição limpa de if/elif/else em cascata
def cmd_salvar(dados): return f'Dados salvos: {dados}'
def cmd_deletar(dados): return f'Dados deletados: {dados}'
def cmd_exportar(dados): return f'Exportando {dados} para PDF'

dispatcher = {
    'save': cmd_salvar,
    'delete': cmd_deletar,
    'export': cmd_exportar
}

acao_solicitada = 'export'
executor = dispatcher.get(acao_solicitada, lambda d: 'Comando inválido')
print('Resultado Dispatcher:', executor('Relatório_Financeiro.xlsx'))

# 5.2 Higher-Order Functions (Funções que manipulam funções)
def aplicar_transformacao(lista, funcao_transformadora):
    return [funcao_transformadora(item) for item in lista]

def elevar_cubo(n): return n ** 3

valores = [1, 2, 3, 4]
print('\nHigher-order aplicada:', aplicar_transformacao(valores, elevar_cubo))

## 6. Funções Aninhadas e Closures

Uma **Closure** ocorre quando uma função aninhada (*inner function*) referencia variáveis do escopo da função que a criou (*enclosing function*), mesmo após a função pai ter finalizado sua execução.

In [ ]:
# Fábrica de Funções (Function Factory)
def criar_formatador_moeda(simbolo, decimais=2):
    # 'simbolo' e 'decimais' ficam encapsulados na closure da função interna
    def formatar(valor):
        return f'{simbolo} {valor:,.{decimais}f}'
    return formatar

formato_real = criar_formatador_moeda('R$', 2)
formato_dolar = criar_formatador_moeda('US$', 2)
formato_btc = criar_formatador_moeda('₿', 6)

print(formato_real(15420.5))
print(formato_dolar(15420.5))
print(formato_btc(0.04512))

# Inspecionando as células da closure internamente em Python
print('\nCélulas capturadas na closure:', formato_real.__closure__)
print('Valores capturados:', [c.cell_contents for c in formato_real.__closure__])

## 7. Introspecção e Metadados de Funções

Funções em Python contêm um rico conjunto de metadados acessíveis tanto por atributos especiais (*dunder attributes*) quanto pelo módulo `inspect` da biblioteca padrão.

In [ ]:
import inspect

def api_endpoint(usuario_id: int, autenticado: bool = False, *permissoes, timeout: int = 60, **opcoes) -> dict:
    """Simula uma chamada de endpoint de API com autenticação."""
    return {'status': 200}

# 7.1 Atributos Dunder nativos
print('__name__     :', api_endpoint.__name__)
print('__defaults__ :', api_endpoint.__defaults__)    # Defaults posicionais
print('__kwdefaults__:', api_endpoint.__kwdefaults__) # Defaults keyword-only
print('__code__.co_varnames:', api_endpoint.__code__.co_varnames)

# 7.2 Introspecção rica com inspect.signature
sig = inspect.signature(api_endpoint)
print('\n--- Assinatura com inspect.signature ---')
print('Assinatura:', sig)
for nome, p in sig.parameters.items():
    print(f'  - Parâmetro: {nome:<14} | Tipo: {p.kind.name:<20} | Default: {p.default}')

## 8. Recursão e Limite de Pilha

Funções definidas com `def` podem chamar a si mesmas. Toda função recursiva exige:
1. **Caso Base:** Condição de parada que impede loop infinito.
2. **Caso Recursivo:** Chamada que reduz o problema em direção ao caso base.

In [ ]:
import sys

def fatorial(n: int) -> int:
    # Caso base
    if n <= 1:
        return 1
    # Caso recursivo
    return n * fatorial(n - 1)

print('Fatorial de 5:', fatorial(5))
print('Fatorial de 7:', fatorial(7))

# Limite de segurança de recursão do interpretador Python (Call Stack Limit)
print(f'\nLimite atual de recursão no Python: {sys.getrecursionlimit()} chamadas')

## 9. Tabela Resumo e Melhores Práticas

| Recurso / Padrão | Sintaxe | Para que serve? | Dica / Armadilha |
| :--- | :--- | :--- | :--- |
| **Definição Padrão** | `def f(a, b):` | Declara uma função em runtime | Nomeie funções com verbos em `snake_case` |
| **Default Seguro** | `def f(lst=None):` | Garante nova instância em cada chamada | 🚨 NUNCA use `lst=[]` ou `dict={}` como default |
| **Positional-Only** | `def f(a, /, b):` | `a` só pode ser passado por posição | Útil para APIs quando o nome do param pode mudar |
| **Keyword-Only** | `def f(a, *, b):` | `b` é obrigatoriamente nomeado (`b=...`) | Torna chamadas com booleanos autoexplicativas |
| **`*args`** | `def f(*args):` | Recebe $N$ argumentos posicionais em Tupla | Útil para funções agregadoras ou wrappers |
| **`**kwargs`** | `def f(**kwargs):` | Recebe $N$ argumentos nomeados em Dicionário | Ideal para repasse de configurações e decorators |
| **Type Hints** | `def f(x: int) -> str:` | Declara contratos de tipos | Melhora autocomplete e segurança com Mypy |
| **`nonlocal`** | `nonlocal x` | Altera variável do escopo *enclosing* | Permite estado em closures sem variáveis globais |
| **`global`** | `global x` | Altera variável do escopo do módulo | Evite no dia a dia para não criar acoplamento oculto |
| **Introspecção** | `inspect.signature(f)` | Lê parâmetros e defaults dinamicamente | Base para frameworks como FastAPI e Pydantic |